# Zero-Shot LLM Baseline Analysis
## How well does GPT-5.4-mini solve circle packing with no evolution?

100 independent samples from the LLM with a fixed prompt, no evolution loop, no feedback. Establishes a floor for comparison against all HDE variants.

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import stats

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 7)
plt.rcParams['font.size'] = 12

TARGET = 2.635

# Load results
results_path = Path('zero_shot_results.json')
if results_path.exists():
    with open(results_path) as f:
        results = json.load(f)
    print(f'Loaded {len(results)} samples')
else:
    print('No results file found. Run zero_shot_baseline.py first.')
    results = []

## 1. Score Distribution

In [ ]:
if results:
    all_scores = [r['combined_score'] for r in results]
    valid_scores = [s for s in all_scores if s > 0]
    invalid_count = len(all_scores) - len(valid_scores)
    
    fig, axes = plt.subplots(1, 3, figsize=(20, 6))
    
    # All scores including zeros
    axes[0].hist(all_scores, bins=30, edgecolor='black', alpha=0.7, color='steelblue')
    axes[0].set_xlabel('Combined Score')
    axes[0].set_ylabel('Count')
    axes[0].set_title(f'All Samples (n={len(all_scores)})')
    axes[0].axvline(x=np.mean(all_scores), color='red', linestyle='--',
                    label=f'Mean={np.mean(all_scores):.4f}')
    axes[0].legend()
    
    # Valid scores only
    if valid_scores:
        axes[1].hist(valid_scores, bins=25, edgecolor='black', alpha=0.7, color='mediumseagreen')
        axes[1].set_xlabel('Combined Score')
        axes[1].set_ylabel('Count')
        axes[1].set_title(f'Valid Programs Only (n={len(valid_scores)}/{len(all_scores)})')
        axes[1].axvline(x=np.mean(valid_scores), color='red', linestyle='--',
                        label=f'Mean={np.mean(valid_scores):.4f}')
        axes[1].axvline(x=np.median(valid_scores), color='orange', linestyle='--',
                        label=f'Median={np.median(valid_scores):.4f}')
        axes[1].legend()
    
    # Pie chart: valid vs invalid breakdown
    error_types = {}
    for r in results:
        if r['combined_score'] == 0:
            err = r.get('error', 'unknown')
            if 'overlap' in str(err).lower() or 'Circles' in str(err):
                error_types['Overlap violations'] = error_types.get('Overlap violations', 0) + 1
            elif 'Invalid shapes' in str(err) or 'shape' in str(err).lower():
                error_types['Wrong shapes'] = error_types.get('Wrong shapes', 0) + 1
            elif 'run_packing' in str(err):
                error_types['Missing run_packing'] = error_types.get('Missing run_packing', 0) + 1
            elif 'No code' in str(err):
                error_types['No code extracted'] = error_types.get('No code extracted', 0) + 1
            elif err and err != 'unknown':
                error_types['Runtime error'] = error_types.get('Runtime error', 0) + 1
            else:
                error_types['Invalid (other)'] = error_types.get('Invalid (other)', 0) + 1
    
    labels = ['Valid'] + list(error_types.keys())
    sizes = [len(valid_scores)] + list(error_types.values())
    colors_pie = ['mediumseagreen'] + plt.cm.Reds(np.linspace(0.3, 0.8, len(error_types))).tolist()
    axes[2].pie(sizes, labels=labels, autopct='%1.0f%%', colors=colors_pie, startangle=90)
    axes[2].set_title('Program Validity')
    
    plt.tight_layout()
    plt.savefig('zero_shot_distribution.png', dpi=150, bbox_inches='tight')
    plt.show()

## 2. Comparison with Evolutionary Runs

In [ ]:
# Load evolutionary best scores for comparison
import glob, re

def get_final_scores(base_dir, n_runs=10):
    scores = []
    for i in range(1, n_runs + 1):
        info = Path(base_dir) / f'run_{i}' / 'best' / 'best_program_info.json'
        if info.exists():
            with open(info) as f:
                d = json.load(f)
            scores.append(d['metrics'].get('combined_score', 0))
    return scores

comparisons = {
    'Zero-shot\n(single call)': valid_scores if results else [],
    'v0: Baseline\n(100 iters)': get_final_scores('baseline_runs'),
    'v0.2: +T300\n(100 iters)': get_final_scores('baseline_v02_runs'),
    'v1.3: HDE\n(100 iters)': get_final_scores('hde_v13_runs'),
}
# Filter empty
comparisons = {k: v for k, v in comparisons.items() if v}

if comparisons:
    fig, axes = plt.subplots(1, 2, figsize=(18, 7))
    
    # Box plot comparison
    positions = list(range(len(comparisons)))
    bp = axes[0].boxplot(comparisons.values(), positions=positions, widths=0.5, patch_artist=True)
    colors_box = ['#ff7f0e', '#1f77b4', '#17becf', '#2ca02c', '#d62728'][:len(comparisons)]
    for i, box in enumerate(bp['boxes']):
        box.set_facecolor(colors_box[i])
        box.set_alpha(0.6)
    axes[0].set_xticks(positions)
    axes[0].set_xticklabels(comparisons.keys())
    axes[0].set_ylabel('Best Score per Run')
    axes[0].set_title('Zero-Shot vs Evolutionary: Final Score Distribution')
    
    # Swarm overlay
    for i, (name, scores) in enumerate(comparisons.items()):
        jitter = np.random.normal(0, 0.04, len(scores))
        axes[0].scatter([i + j for j in jitter], scores, color=colors_box[i],
                       alpha=0.6, s=30, zorder=3)
    
    # Note: zero-shot shows ALL valid samples, evolutionary shows best-per-run
    axes[0].text(0.02, 0.02, 
                'Note: Zero-shot = all valid samples\nEvolutionary = best score per run (10 runs)',
                transform=axes[0].transAxes, fontsize=9, verticalalignment='bottom',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
    
    # Bar chart: mean comparison
    names = list(comparisons.keys())
    means = [np.mean(s) for s in comparisons.values()]
    stds = [np.std(s) for s in comparisons.values()]
    bars = axes[1].bar(range(len(names)), means, yerr=stds, capsize=8,
                       color=colors_box[:len(names)], alpha=0.7, edgecolor='black')
    axes[1].set_xticks(range(len(names)))
    axes[1].set_xticklabels(names)
    axes[1].set_ylabel('Mean Score')
    axes[1].set_title('Mean Score Comparison')
    for bar, m, s in zip(bars, means, stds):
        axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + s + 0.005,
                    f'{m:.4f}', ha='center', fontsize=11, fontweight='bold')
    
    plt.tight_layout()
    plt.savefig('zero_shot_vs_evolutionary.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    # Statistical comparison: zero-shot best vs evolutionary
    if valid_scores:
        print(f'\nZero-shot best sample: {max(valid_scores):.4f} (sum_radii={max(valid_scores)*TARGET:.4f})')
        print(f'Zero-shot mean (valid): {np.mean(valid_scores):.4f}')
        print(f'Zero-shot would need to sample {100/len(valid_scores):.0f}x to get 1 valid program')
        print(f'\nEvolutionary best-per-run means:')
        for name, scores in comparisons.items():
            if 'Zero' not in name:
                print(f'  {name.strip()}: {np.mean(scores):.4f} +/- {np.std(scores):.4f}')

## 3. Token Cost Analysis

In [ ]:
if results:
    prompt_tokens = [r.get('token_usage', {}).get('prompt_tokens', 0) for r in results]
    comp_tokens = [r.get('token_usage', {}).get('completion_tokens', 0) for r in results]
    total_tokens = [r.get('token_usage', {}).get('total_tokens', 0) for r in results]
    llm_times = [r.get('llm_time', 0) for r in results]
    
    fig, axes = plt.subplots(1, 3, figsize=(20, 6))
    
    # Token distribution
    valid_total = [t for t, s in zip(total_tokens, all_scores) if t > 0]
    if valid_total:
        axes[0].hist(valid_total, bins=30, edgecolor='black', alpha=0.7, color='steelblue')
        axes[0].set_xlabel('Total Tokens per Call')
        axes[0].set_ylabel('Count')
        axes[0].set_title(f'Token Usage Distribution (mean={np.mean(valid_total):.0f})')
        axes[0].axvline(x=np.mean(valid_total), color='red', linestyle='--')
    
    # Score vs tokens
    valid_mask = [s > 0 and t > 0 for s, t in zip(all_scores, total_tokens)]
    if any(valid_mask):
        s_plot = [s for s, m in zip(all_scores, valid_mask) if m]
        t_plot = [t for t, m in zip(total_tokens, valid_mask) if m]
        axes[1].scatter(t_plot, s_plot, alpha=0.6, s=40, color='mediumseagreen', edgecolors='black', linewidth=0.5)
        axes[1].set_xlabel('Total Tokens')
        axes[1].set_ylabel('Combined Score')
        axes[1].set_title('Score vs Token Cost (valid programs)')
        # Correlation
        if len(s_plot) > 5:
            r, p = stats.pearsonr(t_plot, s_plot)
            axes[1].text(0.05, 0.95, f'r={r:.3f}, p={p:.3f}',
                        transform=axes[1].transAxes, fontsize=11, verticalalignment='top',
                        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
    
    # Cumulative cost: score of best sample vs cumulative tokens
    cum_tokens = np.cumsum(total_tokens)
    cum_best = np.maximum.accumulate(all_scores)
    axes[2].plot(cum_tokens / 1000, cum_best, color='steelblue', linewidth=2)
    axes[2].set_xlabel('Cumulative Tokens (thousands)')
    axes[2].set_ylabel('Best Score So Far')
    axes[2].set_title('Best Score vs Cumulative Token Budget')
    axes[2].axhline(y=1.0, color='green', linestyle=':', alpha=0.4, label='Target')
    axes[2].legend()
    
    plt.tight_layout()
    plt.savefig('zero_shot_tokens.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    # Summary
    total_all = sum(total_tokens)
    print(f'Total tokens used: {total_all:,}')
    print(f'Mean tokens/call: {np.mean(total_tokens):.0f} (prompt={np.mean(prompt_tokens):.0f}, completion={np.mean(comp_tokens):.0f})')
    print(f'Total LLM time: {sum(llm_times):.0f}s ({sum(llm_times)/60:.1f} min)')
    print(f'Mean LLM time/call: {np.mean(llm_times):.1f}s')

## 4. Summary

In [ ]:
if results:
    print('=' * 65)
    print('ZERO-SHOT BASELINE SUMMARY')
    print('=' * 65)
    print(f'  Model:           GPT-5.4-mini')
    print(f'  Samples:         {len(results)}')
    print(f'  Valid programs:  {len(valid_scores)}/{len(results)} ({100*len(valid_scores)/len(results):.1f}%)')
    if valid_scores:
        print(f'  Score (valid):   {np.mean(valid_scores):.4f} +/- {np.std(valid_scores):.4f}')
        print(f'  Score (all):     {np.mean(all_scores):.4f} +/- {np.std(all_scores):.4f}')
        print(f'  Best score:      {max(valid_scores):.4f} (sum_radii={max(valid_scores)*TARGET:.4f})')
        print(f'  Median (valid):  {np.median(valid_scores):.4f}')
    total_tok = sum(r.get('token_usage', {}).get('total_tokens', 0) for r in results)
    print(f'  Total tokens:    {total_tok:,}')
    print(f'  Tokens/sample:   {total_tok/len(results):,.0f}')
    
    print(f'\n  Key insight:')
    if valid_scores:
        evo_mean = np.mean(get_final_scores('baseline_runs')) if get_final_scores('baseline_runs') else 0
        zs_mean = np.mean(valid_scores)
        if evo_mean > 0:
            gap = evo_mean - zs_mean
            print(f'  Evolutionary mean: {evo_mean:.4f}')
            print(f'  Zero-shot mean:    {zs_mean:.4f}')
            print(f'  Gap:               {gap:+.4f} ({gap/zs_mean*100:+.1f}%)')
            if gap > 0.05:
                print(f'  -> Evolution adds significant value over zero-shot sampling')
            elif gap > 0:
                print(f'  -> Evolution adds modest value over zero-shot sampling')
            else:
                print(f'  -> Evolution does NOT beat zero-shot -- just sampling!')